In [ ]:
#pip install imageio

In [ ]:
# PCA
# 画出解释方差曲线, 找到合适的保留大部分信息的拐点, 然后将这个点作为输入数据的341维度的目标降维维度

# 数据分割\

# 训练标签保存要方便加载

# 训练后也加入AUCPR

# 数据集划分验证

In [ ]:
# 导入必要的库
import os
import time
from fastkan import *
from fastkan import FastKAN
import random
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
import matplotlib.pyplot as plt
import matplotlib.patches as mpts
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, recall_score, cohen_kappa_score, accuracy_score
from sklearn.metrics import precision_score,precision_recall_curve, auc, recall_score, f1_score, accuracy_score, confusion_matrix

from sklearn.preprocessing import minmax_scale
import pandas as pd
from scipy.io import loadmat
from tqdm.notebook import tqdm
from IPython import display
import h5py
import copy
import sys
import glob
import seaborn as sns
from datetime import datetime
%matplotlib inline

In [ ]:
## 超参数和实验设置配置单元格

# 设置随机种子，确保实验可重复性
RANDOM_SEED = 666

LABEL_ID = 1  # 可以根据需要修改

# 数据预处理参数
APPLY_PCA = True   # 是否应用PCA降维
N_PCA = 0          # 设为0表示自动选择主成分数量，大于0表示使用指定数量
NORM = False        # 是否对数据进行标准化/归一化处理


# 定义模型名称，用于结果保存和模型标识
MODEL_NAME = 'Multiclass102_1DKAN'

# 指定数据集名称
DATASET = 'BrainVoxel'


# 训练参数
EPOCH = 100        # 总训练轮数
VAL_EPOCH = 1      # 每隔多少轮进行一次验证
LR = 0.001         # 学习率
WEIGHT_DECAY = 1e-6  # 权重衰减系数，用于L2正则化
BATCH_SIZE = 640    # 批处理大小，固定不变

# 计算设备选择
DEVICE = 0         # -1表示使用CPU，0表示使用第一块GPU(cuda:0)

# 数据参数
FEATURE_DIM = 341  # 输入特征维度
NUM_CLASS = 102      # 二分类问题：正类和负类
FIXED_GRID = 10     # 固定网格大小，不进行网格扩展


# 模型检查点路径
CHECK_POINT = None  # 加载预训练模型的路径，None表示从头开始训练

# 结果保存路径
SAVE_PATH = f"./Results/{MODEL_NAME}/{DATASET}"
# 如果保存目录不存在，则创建该目录
if not os.path.isdir(SAVE_PATH):
    os.makedirs(SAVE_PATH)

In [ ]:
# ## 设置随机数种子，确保实验结果可复现

# # 为Python的random模块设置随机种子
# random.seed(RANDOM_SEED)

# # 为PyTorch的CPU操作设置随机种子
# torch.manual_seed(RANDOM_SEED)

# # 为当前GPU设置随机种子
# torch.cuda.manual_seed(RANDOM_SEED)

# # 为所有可用GPU设置相同的随机种子
# torch.cuda.manual_seed_all(RANDOM_SEED)

# # 为NumPy库设置随机种子
# np.random.seed(RANDOM_SEED)

# # 禁用CuDNN的非确定性算法
# torch.backends.cudnn.deterministic = True

# # 禁用CuDNN的自动优化选择
# torch.backends.cudnn.benchmark = False

In [ ]:
def analyze_pca_variance(X, max_components=None, plot=True, save_path=None):
    """
    分析PCA的方差解释率，找到合适的降维维度
    
    参数:
        X (ndarray): 输入数据
        max_components (int): 最大考虑的主成分数，None表示使用特征维度
        plot (bool): 是否绘制解释方差曲线
        save_path (str): 保存图像的路径，None表示不保存
        
    返回:
        optimal_n_components: 建议的主成分数量
    """
    # 确定最大主成分数
    if max_components is None:
        max_components = min(X.shape[0], X.shape[1])
    else:
        max_components = min(max_components, X.shape[0], X.shape[1])
    
    # 计算所有可能的主成分
    pca = PCA(n_components=max_components)
    pca.fit(X)
    
    # 计算累积解释方差
    explained_variance_ratio = pca.explained_variance_ratio_
    cumulative_variance_ratio = np.cumsum(explained_variance_ratio)
    
    # 寻找方差解释率达到95%的拐点
    threshold = 0.95
    optimal_n_components = np.argmax(cumulative_variance_ratio >= threshold) + 1
    
    # 寻找拐点（斜率变化最大的点）
    gradient = np.gradient(explained_variance_ratio)
    gradient_of_gradient = np.gradient(gradient)
    elbow_index = np.argmax(np.abs(gradient_of_gradient))
    elbow_n_components = elbow_index + 1
    
    if plot:
        plt.figure(figsize=(12, 6))
    
        # Plot Explained Variance Ratio
        plt.subplot(1, 2, 1)
        plt.plot(range(1, len(explained_variance_ratio) + 1), 
                 explained_variance_ratio, 'bo-', markersize=4)
        plt.axvline(x=elbow_n_components, color='r', linestyle='--', 
                    label=f'Elbow Point: {elbow_n_components} Components')
        plt.xlabel('Number of Principal Components')
        plt.ylabel('Explained Variance Ratio')
        plt.title('Explained Variance Ratio per Principal Component')
        plt.grid(True)
        plt.legend()
    
        # Plot Cumulative Explained Variance
        plt.subplot(1, 2, 2)
        plt.plot(range(1, len(cumulative_variance_ratio) + 1), 
                 cumulative_variance_ratio, 'ro-', markersize=4)
        plt.axhline(y=threshold, color='g', linestyle='--', 
                    label=f'{threshold*100}% Variance')
        plt.axvline(x=optimal_n_components, color='b', linestyle='--', 
                    label=f'Threshold Components: {optimal_n_components}')
        plt.xlabel('Number of Principal Components')
        plt.ylabel('Cumulative Explained Variance Ratio')
        plt.title('Cumulative Explained Variance Ratio')
        plt.grid(True)
        plt.legend()
    
        plt.tight_layout()
    
        if save_path:
            plt.savefig(save_path)
        plt.show()
    
    print(f"方差拐点对应的主成分数量: {elbow_n_components}")
    print(f"达到{threshold*100}%方差解释率需要的主成分数量: {optimal_n_components}")
    print(f"前{optimal_n_components}个主成分解释了总方差的{cumulative_variance_ratio[optimal_n_components-1]*100:.2f}%")
    
    # 修改为使用95%阈值点
    suggested_components = optimal_n_components  # 使用保留95%信息的维度
    return suggested_components, explained_variance_ratio, cumulative_variance_ratio

In [ ]:

def get_best_model(metrics_list, epoch_list, save_path, del_others=True):
    """
    根据验证指标找到最佳模型
    
    参数：
        metrics_list: 验证指标列表（如准确率）
        epoch_list: 对应于指标的训练轮数列表
        save_path: 存储模型的目录
        del_others: 是否删除其他模型
        
    返回：
        best_model_path: 最佳模型的路径
    """
    metrics_list = np.array(metrics_list)
    epoch_list = np.array(epoch_list)
    best_index = np.argmax(metrics_list)
    best_epoch = epoch_list[best_index]
    best_metric = metrics_list[best_index]
    
    # 查找匹配最佳轮数和准确率的模型文件
    file_pattern = f"epoch_{best_epoch}_acc_{best_metric:.4f}*.pth"
    matching_files = glob.glob(os.path.join(save_path, file_pattern))
    
    if not matching_files:
        # 如果找不到精确匹配的文件，尝试更宽松的搜索
        print(f"未找到精确匹配的模型文件，正在搜索 epoch {best_epoch} 的模型...")
        file_pattern = f"epoch_{best_epoch}_*.pth"
        matching_files = glob.glob(os.path.join(save_path, file_pattern))
    
    if not matching_files:
        raise FileNotFoundError(f"无法找到 epoch {best_epoch} 的模型文件")
    
    best_model_path = matching_files[0]
    print(f"最佳模型: {os.path.basename(best_model_path)} (准确率: {best_metric:.4f})")
    
    # 可选：删除其他模型文件
    if del_others:
        for f in os.listdir(save_path):
            if f.endswith('.pth') and os.path.join(save_path, f) != best_model_path:
                try:
                    os.remove(os.path.join(save_path, f))
                    print(f"已删除: {f}")
                except:
                    print(f"无法删除: {f}")
    
    return best_model_path


In [ ]:
def analyze_model_features(model, save_path=None, apply_pca_flag=True, pca_model=None):
    """
    分析模型中特征的重要性
    
    参数:
        model: 训练好的KAN模型
        save_path: 保存路径
        apply_pca_flag: 是否应用了PCA
        pca_model: PCA模型，用于特征映射
    """
    # 获取模型输入层的权重
    input_weights = model.kan.layers[0].base_linear.weight.data.cpu().numpy()
    
    # 计算特征的平均绝对权重值（简单的重要性度量）
    feature_importance = np.mean(np.abs(input_weights), axis=0)
    
    # 找出前20个最重要的特征
    top_n = min(20, len(feature_importance))
    top_indices = np.argsort(feature_importance)[-top_n:][::-1]
    top_importance = feature_importance[top_indices]
    
    # 可视化特征重要性
    plt.figure(figsize=(12, 8))
    
    if apply_pca_flag and pca_model is not None:
        feature_names = [f"PC {i+1}" for i in top_indices]
        title = f"Top {top_n} Principal Component Importance"
        
        # 为重要的PC显示其对应的原始特征贡献
        plt.figure(figsize=(15, 10))
        for i, pc_idx in enumerate(top_indices[:5]):  # 只显示前5个最重要的PC
            plt.subplot(5, 1, i+1)
            pc_loadings = pca_model.components_[pc_idx]
            
            # 找出此PC中最重要的原始特征
            top_loading_indices = np.argsort(np.abs(pc_loadings))[-10:]  # 显示前10个
            top_loadings = pc_loadings[top_loading_indices]
            
            plt.bar(range(len(top_loadings)), top_loadings)
            plt.title(f"PC {pc_idx+1} Top Feature Contributions")
            plt.xticks(range(len(top_loadings)), [f"Feature {idx}" for idx in top_loading_indices], rotation=45)
            plt.ylabel("Loading")
        
        plt.tight_layout()
        if save_path:
            base_path, ext = os.path.splitext(save_path)
            pc_loading_path = f"{base_path}_pc_loadings{ext}"
            plt.savefig(pc_loading_path)
        
        # 回到主图
        plt.figure(figsize=(12, 8))
    else:
        feature_names = [f"Feature {i+1}" for i in top_indices]
        title = f"Top {top_n} Feature Importance"
    
    plt.barh(range(top_n), top_importance, align='center')
    plt.yticks(range(top_n), feature_names)
    plt.xlabel('Mean Absolute Weight')
    plt.title(title)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
    
    plt.show()
    
    return feature_importance

In [ ]:
class MulticlassDataset(Dataset):
    """多分类数据集类"""
    def __init__(self, data, labels, is_inference=False):
        """
        初始化数据集
        
        参数:
            data: 特征数据，形状为(n_samples, feature_dim)
            labels: 标签数据，形状为(n_samples,)，值范围为0到NUM_CLASS-1
            is_inference: 是否为推理模式（不返回标签）
        """
        super(MulticlassDataset, self).__init__()
        self.data = data
        self.labels = labels
        self.is_inference = is_inference
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        x = self.data[idx]
        x = torch.FloatTensor(x)
        
        if self.is_inference:
            return x
        else:
            y = self.labels[idx]
            y = torch.LongTensor([int(y)])[0]  # 确保标签是整数
            return x, y

In [ ]:
def load_multiclass_data_from_dirs(data_dirs, apply_pca=True, n_components=24, norm=True):
    """
    从指定的目录加载多类别数据
    
    参数:
        data_dirs: 包含训练集、测试集和验证集路径的字典
        apply_pca: 是否应用PCA降维
        n_components: PCA降维的组件数量
        norm: 是否进行归一化
        
    返回:
        dataset_dict: 包含训练、测试和验证数据的字典
    """
    print(f"正在从目录加载多类别数据...")
    print(f"训练集目录: {data_dirs['train_dir']}")
    print(f"测试集目录: {data_dirs['test_dir']}")
    print(f"验证集目录: {data_dirs['val_dir']}")
    
    # 辅助函数：从目录中加载数据
    def load_dir_data(dir_path):
        data_files = glob.glob(os.path.join(dir_path, "*.npy"))
        features_file = next((f for f in data_files if "features" in f.lower() or "data" in f.lower()), None)
        labels_file = next((f for f in data_files if "label" in f.lower() or "target" in f.lower()), None)
        
        if not features_file or not labels_file:
            # 如果没有找到特定的文件，则尝试使用任何 .npy 文件
            data_files = sorted(data_files)
            if len(data_files) >= 2:
                features_file = data_files[0]  # 假设第一个文件是特征数据
                labels_file = data_files[1]    # 假设第二个文件是标签数据
            else:
                raise FileNotFoundError(f"在 {dir_path} 中找不到数据和标签文件")
        
        print(f"加载特征数据: {os.path.basename(features_file)}")
        print(f"加载标签数据: {os.path.basename(labels_file)}")
        
        features = np.load(features_file)
        labels = np.load(labels_file)
        
        return features, labels
    
    # 从每个目录加载数据
    try:
        train_data, train_labels = load_dir_data(data_dirs['train_dir'])
        test_data, test_labels = load_dir_data(data_dirs['test_dir']) 
        val_data, val_labels = load_dir_data(data_dirs['val_dir'])
    except Exception as e:
        print(f"数据加载错误: {e}")
        raise
    
    # 检查标签范围
    print(f"训练集标签范围: {np.min(train_labels)} 至 {np.max(train_labels)}")
    print(f"测试集标签范围: {np.min(test_labels)} 至 {np.max(test_labels)}")
    print(f"验证集标签范围: {np.min(val_labels)} 至 {np.max(val_labels)}")
    
    # 确保标签在合理范围内 (0 到 NUM_CLASS-1)
    if np.max(train_labels) >= NUM_CLASS or np.min(train_labels) < 0:
        print(f"警告: 训练集标签超出预期范围 [0, {NUM_CLASS-1}]")
    if np.max(test_labels) >= NUM_CLASS or np.min(test_labels) < 0:
        print(f"警告: 测试集标签超出预期范围 [0, {NUM_CLASS-1}]")
    if np.max(val_labels) >= NUM_CLASS or np.min(val_labels) < 0:
        print(f"警告: 验证集标签超出预期范围 [0, {NUM_CLASS-1}]")
    
    # 是否应用PCA降维
    if apply_pca:
        # 将所有数据合并，以便进行PCA训练
        all_data = np.vstack([train_data, test_data, val_data])
        
        if n_components > 0:
            pca_model = PCA(n_components=n_components)
            pca_model.fit(all_data)
        else:
            # 自动选择PCA组件数量
            n_components, _, _ = analyze_pca_variance(all_data, plot=True)
            pca_model = PCA(n_components=n_components)
            pca_model.fit(all_data)
            
        # 应用PCA变换
        train_data = pca_model.transform(train_data)
        test_data = pca_model.transform(test_data)
        val_data = pca_model.transform(val_data)
        
        # 如果需要，进行归一化
        if norm:
            # 计算所有样本的归一化参数
            all_transformed = np.vstack([train_data, test_data, val_data])
            mins = np.min(all_transformed, axis=0)
            maxs = np.max(all_transformed, axis=0)
            ranges = maxs - mins + 1e-10  # 避免除零错误
            
            # 归一化数据
            train_data = (train_data - mins) / ranges
            test_data = (test_data - mins) / ranges
            val_data = (val_data - mins) / ranges
        
        feature_dim = train_data.shape[1]
        print(f"PCA后: 特征维度 = {feature_dim}")
    else:
        feature_dim = train_data.shape[1]
        pca_model = None
        print(f"未应用PCA: 特征维度 = {feature_dim}")
    
    # 输出数据形状及类别统计
    print(f"训练集: {train_data.shape}, 标签: {train_labels.shape}")
    print(f"测试集: {test_data.shape}, 标签: {test_labels.shape}")
    print(f"验证集: {val_data.shape}, 标签: {val_labels.shape}")
    
    # 输出类别分布
    class_distribution = []
    print("\n类别分布:")
    print(f"{'类别':^10}{'训练集':^10}{'测试集':^10}{'验证集':^10}{'总计':^10}")
    print("-" * 50)
    
    for i in range(NUM_CLASS):
        train_count = np.sum(train_labels == i)
        test_count = np.sum(test_labels == i)
        val_count = np.sum(val_labels == i)
        total = train_count + test_count + val_count
        class_distribution.append((i, train_count, test_count, val_count, total))
        print(f"{i:^10}{train_count:^10}{test_count:^10}{val_count:^10}{total:^10}")
    
    # 输出总计数量
    total_train = len(train_labels)
    total_test = len(test_labels)
    total_val = len(val_labels)
    total_all = total_train + total_test + total_val
    print("-" * 50)
    print(f"{'总计':^10}{total_train:^10}{total_test:^10}{total_val:^10}{total_all:^10}")
    
    return {
        'train_samples': train_data,
        'train_labels': train_labels,
        'test_samples': test_data,
        'test_labels': test_labels,
        'val_samples': val_data,
        'val_labels': val_labels,
        'feature_dim': feature_dim,
        'pca_model': pca_model,
        'class_distribution': class_distribution
    }


In [ ]:
def load_merged_dataset(merged_dir, apply_pca=True, pca_model=None, norm=True):
    """
    加载合并后的数据集
    
    参数:
        merged_dir: 合并数据集的路径
        apply_pca: 是否应用PCA降维
        pca_model: 预训练的PCA模型 (如果为None，则训练新的模型)
        norm: 是否进行归一化
        
    返回:
        merged_data: 特征数据
        merged_labels: 标签数据
    """
    print(f"正在从 {merged_dir} 加载合并数据集...")
    
    # 查找数据文件
    data_files = glob.glob(os.path.join(merged_dir, "*.npy"))
    features_file = next((f for f in data_files if "features" in f.lower() or "data" in f.lower()), None)
    labels_file = next((f for f in data_files if "label" in f.lower() or "target" in f.lower()), None)
    
    if not features_file or not labels_file:
        # 如果没有找到特定的文件，则尝试使用任何 .npy 文件
        data_files = sorted(data_files)
        if len(data_files) >= 2:
            features_file = data_files[0]  # 假设第一个文件是特征数据
            labels_file = data_files[1]    # 假设第二个文件是标签数据
        else:
            raise FileNotFoundError(f"在 {merged_dir} 中找不到数据和标签文件")
    
    print(f"加载特征数据: {os.path.basename(features_file)}")
    print(f"加载标签数据: {os.path.basename(labels_file)}")
    
    merged_data = np.load(features_file)
    merged_labels = np.load(labels_file)
    
    # 检查标签范围
    print(f"合并数据集标签范围: {np.min(merged_labels)} 至 {np.max(merged_labels)}")
    
    # 是否应用PCA降维
    if apply_pca and pca_model is not None:
        merged_data = pca_model.transform(merged_data)
        print(f"已使用预训练PCA模型: 特征维度 = {merged_data.shape[1]}")
        
        # 如果需要，进行归一化
        if norm:
            mins = np.min(merged_data, axis=0)
            maxs = np.max(merged_data, axis=0)
            ranges = maxs - mins + 1e-10  # 避免除零错误
            merged_data = (merged_data - mins) / ranges
    elif apply_pca:
        # 训练新的PCA模型
        if N_PCA > 0:
            pca_model = PCA(n_components=N_PCA)
        else:
            n_components, _, _ = analyze_pca_variance(merged_data, plot=True)
            pca_model = PCA(n_components=n_components)
        
        pca_model.fit(merged_data)
        merged_data = pca_model.transform(merged_data)
        print(f"已应用新训练的PCA: 特征维度 = {merged_data.shape[1]}")
        
        # 如果需要，进行归一化
        if norm:
            mins = np.min(merged_data, axis=0)
            maxs = np.max(merged_data, axis=0)
            ranges = maxs - mins + 1e-10  # 避免除零错误
            merged_data = (merged_data - mins) / ranges
    
    # 输出数据形状及类别统计
    print(f"合并数据集: {merged_data.shape}, 标签: {merged_labels.shape}")
    
    # 输出类别分布
    print("\n合并数据集类别分布:")
    class_counts = []
    for i in range(NUM_CLASS):
        count = np.sum(merged_labels == i)
        class_counts.append(count)
        print(f"类别 {i}: {count} 个样本 ({count/len(merged_labels)*100:.2f}%)")
    
    return merged_data, merged_labels


In [ ]:
def plot_confusion_matrix(conf_matrix, class_names=None, figsize=(20, 16), save_path=None):
    """
    绘制混淆矩阵
    
    参数:
        conf_matrix: 混淆矩阵
        class_names: 类别名称列表，默认为None（使用索引）
        figsize: 图像大小
        save_path: 保存路径，None表示不保存
    """
    if class_names is None:
        class_names = [str(i) for i in range(conf_matrix.shape[0])]
    
    plt.figure(figsize=figsize)
    
    # 对于类别数量很多的情况，使用log尺度更合适
    if conf_matrix.shape[0] > 10:
        # 添加一个小值，避免log(0)
        plt.imshow(np.log1p(conf_matrix), cmap='Blues')
        plt.colorbar(label='log(frequency)')
    else:
        plt.imshow(conf_matrix, cmap='Blues')
        plt.colorbar(label='frequency')
    
    # 添加类别标签
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=90)
    plt.yticks(tick_marks, class_names)
    
    # 添加文本
    thresh = conf_matrix.max() / 2.
    for i in range(conf_matrix.shape[0]):
        for j in range(conf_matrix.shape[1]):
            if conf_matrix[i, j] > 0:
                text_color = "white" if conf_matrix[i, j] > thresh else "black"
                plt.text(j, i, format(conf_matrix[i, j], 'd'),
                         ha="center", va="center", color=text_color)
    
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()

In [ ]:
def evaluate_on_all_datasets_detailed(model, train_loader, test_loader, val_loader, merged_loader, device, save_path):
    """
    对所有数据集（训练集、测试集、验证集、合并集）进行详细评估
    
    参数:
        model: 训练好的模型
        train_loader: 训练数据加载器
        test_loader: 测试数据加载器
        val_loader: 验证数据加载器
        merged_loader: 合并数据加载器
        device: 计算设备
        save_path: 结果保存路径
        
    返回:
        包含所有评估结果的字典
    """
    print("开始对所有数据集进行综合评估...")
    
    # 创建结果存储目录
    results_dir = os.path.join(save_path, "evaluation_results")
    os.makedirs(results_dir, exist_ok=True)
    
    # 评估训练集
    print("\n" + "="*60)
    print("正在评估训练集...")
    train_results = evaluate_multiclass_model(model, train_loader, device)
    
    # 评估测试集
    print("\n" + "="*60)
    print("正在评估测试集...")
    test_results = evaluate_multiclass_model(model, test_loader, device)
    
    # 评估验证集
    print("\n" + "="*60)
    print("正在评估验证集...")
    val_results = evaluate_multiclass_model(model, val_loader, device)
    
    # 评估合并集
    print("\n" + "="*60)
    print("正在评估合并数据集...")
    merged_results = evaluate_multiclass_model(model, merged_loader, device)
    
    # 打印评估摘要
    print("\n" + "="*60)
    print("评估摘要")
    print("="*60)
    print(f"{'数据集':^15}{'准确率':^15}{'表现最佳类别':^30}{'表现较差类别':^30}")
    print("-"*90)
    
    # 获取表现最佳和较差的类别（前 3 名和后 3 名）
    def get_top_bottom_classes(per_class_acc):
        top_idx = np.argsort(per_class_acc)[-3:][::-1]
        bottom_idx = np.argsort(per_class_acc)[:3]
        top_str = ", ".join([f"{i}({per_class_acc[i]:.4f})" for i in top_idx])
        bottom_str = ", ".join([f"{i}({per_class_acc[i]:.4f})" for i in bottom_idx])
        return top_str, bottom_str
    
    # 获取并打印各数据集的最佳/最差类别
    train_top, train_bottom = get_top_bottom_classes(train_results['per_class_accuracy'])
    test_top, test_bottom = get_top_bottom_classes(test_results['per_class_accuracy'])
    val_top, val_bottom = get_top_bottom_classes(val_results['per_class_accuracy'])
    merged_top, merged_bottom = get_top_bottom_classes(merged_results['per_class_accuracy'])
    
    print(f"{'训练集':^15}{train_results['accuracy']:.4f}^15{train_top:^30}{train_bottom:^30}")
    print(f"{'测试集':^15}{test_results['accuracy']:.4f}^15{test_top:^30}{test_bottom:^30}")
    print(f"{'验证集':^15}{val_results['accuracy']:.4f}^15{val_top:^30}{val_bottom:^30}")
    print(f"{'合并集':^15}{merged_results['accuracy']:.4f}^15{merged_top:^30}{merged_bottom:^30}")

        # 创建并保存可视化图表
    
    # 1. 绘制准确率对比图
    plt.figure(figsize=(12, 8))
    plt.suptitle("各数据集性能对比", fontsize=16)
    
    # 总体准确率对比
    plt.subplot(2, 2, 1)
    datasets = ['训练集', '测试集', '验证集', '合并集']
    accuracies = [
        train_results['accuracy'], 
        test_results['accuracy'], 
        val_results['accuracy'],
        merged_results['accuracy']
    ]
    
    plt.bar(datasets, accuracies, color=['blue', 'green', 'orange', 'red'])
    plt.ylabel('准确率')
    plt.title('总体准确率对比')
    plt.ylim(0, 1.0)  # 设置 y 轴范围 0-1
    plt.grid(axis='y')
    
    # 添加准确率数值
    for i, v in enumerate(accuracies):
        plt.text(i, v + 0.02, f"{v:.4f}", ha='center')
    
    # 2. 绘制类别样本分布
    plt.subplot(2, 2, 2)
    
    # 获取每个数据集的类别样本数
    train_counts = np.bincount(train_loader.dataset.labels, minlength=NUM_CLASS)
    test_counts = np.bincount(test_loader.dataset.labels, minlength=NUM_CLASS)
    val_counts = np.bincount(val_loader.dataset.labels, minlength=NUM_CLASS)
    merged_counts = np.bincount(merged_loader.dataset.labels, minlength=NUM_CLASS)
    
    # 仅绘制前 20 个类别以保持可读性
    plot_classes = min(20, NUM_CLASS)
    class_indices = np.arange(plot_classes)
    
    plt.bar(class_indices - 0.3, train_counts[:plot_classes], width=0.2, label='训练集', alpha=0.7)
    plt.bar(class_indices - 0.1, test_counts[:plot_classes], width=0.2, label='测试集', alpha=0.7)
    plt.bar(class_indices + 0.1, val_counts[:plot_classes], width=0.2, label='验证集', alpha=0.7)
    plt.bar(class_indices + 0.3, merged_counts[:plot_classes], width=0.2, label='合并集', alpha=0.7)
    
    plt.xlabel('类别')
    plt.ylabel('样本数量')
    plt.title(f'样本分布 (前 {plot_classes} 个类别)')
    plt.xticks(class_indices)
    plt.legend()
    
    # 3. 绘制每类准确率
    plt.subplot(2, 2, 3)
    
    # 仅绘制前 20 个类别以保持可读性
    plt.plot(range(plot_classes), train_results['per_class_accuracy'][:plot_classes], 'b-', label='训练集')
    plt.plot(range(plot_classes), test_results['per_class_accuracy'][:plot_classes], 'g-', label='测试集')
    plt.plot(range(plot_classes), val_results['per_class_accuracy'][:plot_classes], 'orange', label='验证集')
    plt.plot(range(plot_classes), merged_results['per_class_accuracy'][:plot_classes], 'r-', label='合并集')
    
    plt.xlabel('类别')
    plt.ylabel('准确率')
    plt.title(f'每类准确率 (前 {plot_classes} 个类别)')
    plt.xticks(range(plot_classes))
    plt.grid(True)
    plt.legend()
    
    # 4. 绘制准确率分布直方图
    plt.subplot(2, 2, 4)
    
    plt.hist(train_results['per_class_accuracy'], bins=20, alpha=0.5, label='训练集')
    plt.hist(test_results['per_class_accuracy'], bins=20, alpha=0.5, label='测试集')
    plt.hist(val_results['per_class_accuracy'], bins=20, alpha=0.5, label='验证集')
    plt.hist(merged_results['per_class_accuracy'], bins=20, alpha=0.5, label='合并集')
    
    plt.xlabel('准确率')
    plt.ylabel('类别数量')
    plt.title('各类别准确率分布')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(os.path.join(results_dir, 'all_datasets_performance.png'), dpi=300)
    print(f"性能对比图已保存至: {os.path.join(results_dir, 'all_datasets_performance.png')}")
    plt.show()

    # 保存各数据集的混淆矩阵
    print("\n正在保存混淆矩阵...")
    for dataset_name, results in [
        ('训练集', train_results), 
        ('测试集', test_results), 
        ('验证集', val_results),
        ('合并集', merged_results)
    ]:
        cm_file = os.path.join(results_dir, f'{dataset_name}_confusion_matrix.png')
        plot_confusion_matrix(
            results['confusion_matrix'],
            class_names=None,
            figsize=(20, 16),
            save_path=cm_file
        )
        print(f"{dataset_name} 的混淆矩阵已保存至: {cm_file}")
    
    # 生成并保存详细评估报告
    print("\n正在生成详细性能报告...")
    
    # 关键指标的综合报告
    combined_report = f"""# 综合评估报告 - {NUM_CLASS} 类 KAN 模型

## 概览
- 模型名称: {MODEL_NAME}
- 数据集: {DATASET}
- 类别数量: {NUM_CLASS}
- 评估日期: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## 主要性能指标

| 数据集    | 准确率 | 宏平均 F1 | 加权 F1 | 样本数 |
|------------|----------|----------|-------------|---------|
| 训练集   | {train_results['accuracy']:.4f} | {f1_score(train_results['targets'], train_results['predictions'], average='macro'):.4f} | {f1_score(train_results['targets'], train_results['predictions'], average='weighted'):.4f} | {len(train_results['targets'])} |
| 测试集    | {test_results['accuracy']:.4f} | {f1_score(test_results['targets'], test_results['predictions'], average='macro'):.4f} | {f1_score(test_results['targets'], test_results['predictions'], average='weighted'):.4f} | {len(test_results['targets'])} |
| 验证集 | {val_results['accuracy']:.4f} | {f1_score(val_results['targets'], val_results['predictions'], average='macro'):.4f} | {f1_score(val_results['targets'], val_results['predictions'], average='weighted'):.4f} | {len(val_results['targets'])} |
| 合并集     | {merged_results['accuracy']:.4f} | {f1_score(merged_results['targets'], merged_results['predictions'], average='macro'):.4f} | {f1_score(merged_results['targets'], merged_results['predictions'], average='weighted'):.4f} | {len(merged_results['targets'])} |

## 表现最佳的前 5 类别

    """

    # 添加每个数据集的前 5 类别
    for dataset_name, results in [
        ('训练集', train_results), 
        ('测试集', test_results), 
        ('验证集', val_results),
        ('合并集', merged_results)
    ]:
        top5_idx = np.argsort(results['per_class_accuracy'])[-5:][::-1]
        combined_report += f"### {dataset_name} 数据集\n"
        for i, idx in enumerate(top5_idx):
            combined_report += f"- 类别 {idx}: {results['per_class_accuracy'][idx]:.4f}\n"
        combined_report += "\n"
    
    combined_report += "## 表现最差的 5 类别\n\n"
    
    # 添加每个数据集的后 5 类别
    for dataset_name, results in [
        ('训练集', train_results), 
        ('测试集', test_results), 
        ('验证集', val_results),
        ('合并集', merged_results)
    ]:
        bottom5_idx = np.argsort(results['per_class_accuracy'])[:5]
        combined_report += f"### {dataset_name} 数据集\n"
        for i, idx in enumerate(bottom5_idx):
            combined_report += f"- 类别 {idx}: {results['per_class_accuracy'][idx]:.4f}\n"
        combined_report += "\n"
    
    # 保存综合评估报告
    combined_report_file = os.path.join(results_dir, 'combined_evaluation_report.md')
    with open(combined_report_file, 'w') as f:
        f.write(combined_report)
    print(f"综合评估报告已保存至: {combined_report_file}")
    
    # 保存各个数据集的详细报告
    for dataset_name, results in [
        ('训练集', train_results), 
        ('测试集', test_results), 
        ('验证集', val_results),
        ('合并集', merged_results)
    ]:
        detailed_report = f"""# 详细评估报告 - {dataset_name.title()} 数据集

## 概览
- 模型名称: {MODEL_NAME}
- 数据集: {DATASET} ({dataset_name})
- 类别数量: {NUM_CLASS}
- 评估日期: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## 主要性能指标
- 准确率: {results['accuracy']:.4f}
- 宏平均 F1 分数: {f1_score(results['targets'], results['predictions'], average='macro'):.4f}
- 加权 F1 分数: {f1_score(results['targets'], results['predictions'], average='weighted'):.4f}
- 样本数: {len(results['targets'])}

## 混淆矩阵
混淆矩阵已保存为图片文件。

    ## 每类别性能

| 类别 | 样本数 | 精确率 | 召回率 | F1-分数 | 准确率 |
|-------|---------|-----------|--------|----------|----------|
"""
        
        # 添加每个类别的指标
        for i in range(NUM_CLASS):
            # 获取精确率、召回率和 F1 分数
            class_precision = precision_score(results['targets'], results['predictions'], labels=[i], average=None)[0] if i in results['predictions'] else 0
            class_recall = recall_score(results['targets'], results['predictions'], labels=[i], average=None)[0] if i in np.unique(results['targets']) else 0
            class_f1 = f1_score(results['targets'], results['predictions'], labels=[i], average=None)[0] if (i in results['predictions'] and i in np.unique(results['targets'])) else 0
            class_samples = np.sum(np.array(results['targets']) == i)
            
            detailed_report += f"| {i} | {class_samples} | {class_precision:.4f} | {class_recall:.4f} | {class_f1:.4f} | {results['per_class_accuracy'][i]:.4f} |\n"
        
        # 保存详细评估报告
        detailed_report_file = os.path.join(results_dir, f'{dataset_name}_detailed_report.md')
        with open(detailed_report_file, 'w') as f:
            f.write(detailed_report)
        print(f"{dataset_name} 数据集的详细报告已保存至: {detailed_report_file}")
    
    print("\n评估完成！所有结果和可视化已保存。")
    
    return {
        'train_results': train_results,
        'test_results': test_results,
        'val_results': val_results,
        'merged_results': merged_results
    }


In [ ]:
class MulticlassKAN(nn.Module):
    """
    用于多分类的KAN模型
    """
    def __init__(self, input_dim, hidden_dim, num_classes, grid_size=10):
        """
        初始化模型
        
        参数:
            input_dim: 输入特征维度
            hidden_dim: 隐藏层维度
            num_classes: 类别数量
            grid_size: 网格大小
        """
        super(MulticlassKAN, self).__init__()
        
        self.kan = FastKAN(
            layers_hidden=[input_dim, hidden_dim, num_classes],
            num_grids=grid_size
        )
    
    def forward(self, x):
        """
        前向传播
        
        参数:
            x: 输入特征，形状为(batch_size, input_dim)
            
        返回:
            output: 模型输出，形状为(batch_size, num_classes)
        """
        return self.kan(x)

In [ ]:
def train_multiclass_kan(model, train_loader, test_loader, criterion, optimizer, device, 
                        num_epochs=100, val_epoch=1, save_path="./Results"):
    """
    训练多分类KAN模型
    
    参数:
        model: KAN模型
        train_loader: 训练数据加载器
        test_loader: 测试数据加载器
        criterion: 损失函数
        optimizer: 优化器
        device: 计算设备
        num_epochs: 训练轮数
        val_epoch: 验证频率
        save_path: 模型保存路径
    
    返回:
        训练结果统计信息
    """
    # 初始化统计变量
    loss_list = []
    acc_list = []
    val_acc_list = []
    val_epoch_list = []
    
    # 保存起始时间
    train_st = time.time()
    
    # 计算批次数量和样本数量
    batch_num = len(train_loader)
    train_num = len(train_loader.dataset)
    test_num = len(test_loader.dataset)
    
    try:
        # 训练循环
        for e in tqdm(range(num_epochs), desc="Training:"):
            # 设置模型为训练模式
            model.train()
            avg_loss = 0.0
            train_acc = 0
            
            # 批次循环
            for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=batch_num):
                # 将数据移动到指定设备
                data, target = data.to(device), target.to(device)
                
                # 前向传播
                optimizer.zero_grad()
                out = model(data)
                loss = criterion(out, target)
                
                # 反向传播
                loss.backward()
                optimizer.step()
                
                # 累计损失和准确率
                avg_loss += loss.item()
                _, pred = torch.max(out, dim=1)
                train_acc += (pred == target).sum().item()
            
            # 计算本轮平均损失和准确率
            loss_list.append(avg_loss / train_num)
            acc_list.append(train_acc / train_num)
            print(f"epoch {e}/{num_epochs} loss:{loss_list[-1]}  acc:{acc_list[-1]}")
            
            # 验证阶段
            if (e+1) % val_epoch == 0 or (e+1) == num_epochs:
                val_acc = 0
                model.eval()
                
                # 收集验证数据的预测结果
                all_preds = []
                all_targets = []
                
                with torch.no_grad():
                    for batch_idx, (data, target) in tqdm(enumerate(test_loader), total=len(test_loader)):
                        data, target = data.to(device), target.to(device)
                        out = model(data)
                        _, pred = torch.max(out, dim=1)
                        
                        all_preds.extend(pred.cpu().numpy())
                        all_targets.extend(target.cpu().numpy())
                        val_acc += (pred == target).sum().item()
                
                # 计算验证准确率
                val_accuracy = val_acc / test_num
                
                # 保存验证结果
                val_acc_list.append(val_accuracy)
                val_epoch_list.append(e)
                
                # 显示验证指标
                print(f"epoch {e}/{num_epochs}  val_acc:{val_accuracy:.4f}")
                
                # 创建混淆矩阵
                conf_matrix = confusion_matrix(all_targets, all_preds, labels=range(NUM_CLASS))
                
                # 计算每个类别的准确率
                per_class_accuracy = conf_matrix.diagonal() / conf_matrix.sum(axis=1)
                print(f"每个类别的准确率:")
                for i, acc in enumerate(per_class_accuracy):
                    print(f"  类别 {i}: {acc:.4f}")
                
                # 保存当前模型
                save_name = os.path.join(save_path, f"epoch_{e}_acc_{val_accuracy:.4f}.pth")
                save_dict = {
                    'state_dict': model.state_dict(), 
                    'epoch': e+1, 
                    'optimizer': optimizer.state_dict(),
                    'loss_list': loss_list, 
                    'acc_list': acc_list, 
                    'val_acc_list': val_acc_list, 
                    'val_epoch_list': val_epoch_list
                }
                torch.save(save_dict, save_name)
                
    except Exception as exc:
        print(exc)
        import traceback
        traceback.print_exc()
        
    finally:
        print(f'训练停止于epoch {e}')
    
    # 计算总训练时间
    train_time = time.time() - train_st
    print(f"训练时间: {train_time}秒")
    
    # 返回训练结果
    return {
        'loss_list': loss_list,
        'acc_list': acc_list,
        'val_acc_list': val_acc_list,
        'val_epoch_list': val_epoch_list,
        'train_time': train_time
    }

def evaluate_multiclass_model(model, data_loader, device):
    """
    评估多分类模型性能
    
    参数:
        model: 训练好的模型
        data_loader: 数据加载器
        device: 计算设备
    
    返回:
        评估结果
    """
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for data, target in tqdm(data_loader, desc="评估中"):
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, preds = torch.max(output, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
    
    # 计算评估指标
    accuracy = accuracy_score(all_targets, all_preds)
    
    # 计算每个类别的准确率、精确率和召回率
    conf_matrix = confusion_matrix(all_targets, all_preds, labels=range(NUM_CLASS))
    per_class_accuracy = conf_matrix.diagonal() / conf_matrix.sum(axis=1)
    
    # 生成分类报告
    report = classification_report(all_targets, all_preds, labels=range(NUM_CLASS), digits=4)
    
    return {
        'accuracy': accuracy,
        'per_class_accuracy': per_class_accuracy,
        'report': report,
        'predictions': all_preds,
        'targets': all_targets,
        'confusion_matrix': conf_matrix
    }

In [ ]:
# Data directories
DATA_DIRS = {
    'train_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/train",
    'test_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/test",
    'val_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/val"
}
MERGED_DIR = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/merged"

# Load data from directories
print("Loading data from directories...")
dataset_dict = load_multiclass_data_from_dirs(
    DATA_DIRS, 
    apply_pca=APPLY_PCA, 
    n_components=N_PCA, 
    norm=NORM
)

# Save PCA model for later use
pca_model = dataset_dict.get('pca_model')
feature_dim = dataset_dict['feature_dim']

if pca_model is not None:
    import pickle
    pca_save_path = os.path.join(SAVE_PATH, f'pca_model_components_{feature_dim}.pkl')
    with open(pca_save_path, 'wb') as f:
        pickle.dump(pca_model, f)
    print(f"PCA model saved to: {pca_save_path}")

# Create datasets
train_dataset = MulticlassDataset(dataset_dict['train_samples'], dataset_dict['train_labels'])
test_dataset = MulticlassDataset(dataset_dict['test_samples'], dataset_dict['test_labels'])
val_dataset = MulticlassDataset(dataset_dict['val_samples'], dataset_dict['val_labels'])

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Load merged dataset
print("\nLoading merged dataset...")
merged_data, merged_labels = load_merged_dataset(
    MERGED_DIR,
    apply_pca=APPLY_PCA,
    pca_model=pca_model,
    norm=NORM
)
merged_dataset = MulticlassDataset(merged_data, merged_labels)
merged_loader = DataLoader(merged_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Set computing device
device = torch.device(f"cuda:{DEVICE}" if DEVICE>=0 and torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create model
model = MulticlassKAN(feature_dim, 64, NUM_CLASS, FIXED_GRID).to(device)

# Print model architecture
summary(model)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Create training info directory
training_info_dir = os.path.join(SAVE_PATH, "training_info")
os.makedirs(training_info_dir, exist_ok=True)

# Train model
print("\nStarting model training...")
training_results = train_multiclass_kan(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader, 
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=EPOCH,
    val_epoch=VAL_EPOCH,
    save_path=SAVE_PATH
)

# Plot and save training curves
plt.figure(figsize=(15, 5))

# Loss curve
plt.subplot(1, 3, 1)
plt.plot(range(len(training_results['loss_list'])), training_results['loss_list'])
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)

# Accuracy curves
plt.subplot(1, 3, 2)
plt.plot(range(len(training_results['acc_list'])), training_results['acc_list'], label='Train Acc')
plt.plot(training_results['val_epoch_list'], training_results['val_acc_list'], label='Val Acc')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Learning curve (training progress)
plt.subplot(1, 3, 3)
epoch_markers = [0] + [e for e in training_results['val_epoch_list'] if e % 10 == 0 or e == training_results['val_epoch_list'][-1]]
train_acc_markers = [training_results['acc_list'][e] for e in epoch_markers]
val_acc_markers = []

for e in epoch_markers:
    if e in training_results['val_epoch_list']:
        idx = training_results['val_epoch_list'].index(e)
        val_acc_markers.append(training_results['val_acc_list'][idx])
    else:
        val_acc_markers.append(None)

plt.plot(epoch_markers, train_acc_markers, 'bo-', label='Train')
plt.plot(epoch_markers, val_acc_markers, 'ro-', label='Validation')
plt.title('Learning Curve')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
curves_path = os.path.join(training_info_dir, 'training_curves.png')
plt.savefig(curves_path, dpi=300)
print(f"Training curves saved to: {curves_path}")
plt.show()

# Get the best model
best_model_path = get_best_model(
    training_results['val_acc_list'],
    training_results['val_epoch_list'],
    SAVE_PATH,
    del_others=False  # Keep all models for now
)

# 加载最佳模型
best_model = MulticlassKAN(feature_dim, 64, NUM_CLASS, FIXED_GRID).to(device)
model_checkpoint = torch.load(best_model_path, map_location=device)
best_model.load_state_dict(model_checkpoint['state_dict'])
best_model.eval()
print(f"最佳模型已加载: {best_model_path}")

# 对所有数据集进行综合评估
print("\n正在对所有数据集进行综合评估...")
evaluation_results = evaluate_on_all_datasets_detailed(
    best_model,
    train_loader,
    test_loader,
    val_loader,
    merged_loader,
    device,
    SAVE_PATH
)

# 保存模型训练和评估摘要
summary_report = f"""# 102 类 KAN 模型 - 训练与评估摘要

## 模型信息
- 模型名称: {MODEL_NAME}
- 数据集: {DATASET}
- 类别数量: {NUM_CLASS}
- 特征维度: {feature_dim} {'(PCA 降维后)' if APPLY_PCA else ''}
- 固定网格大小: {FIXED_GRID}

## 训练参数
- 学习率: {LR}
- 权重衰减: {WEIGHT_DECAY}
- 批量大小: {BATCH_SIZE}
- 训练轮数: {len(training_results['loss_list'])}
- 计算设备: {device}

## 训练结果
- 最终训练准确率: {training_results['acc_list'][-1]:.4f}
- 最佳验证准确率: {max(training_results['val_acc_list']):.4f} (第 {training_results['val_epoch_list'][np.argmax(training_results['val_acc_list'])]} 轮)
- 训练时间: {training_results['train_time']:.2f} 秒

## 评估结果
- 训练集准确率: {evaluation_results['train_results']['accuracy']:.4f}
- 测试集准确率: {evaluation_results['test_results']['accuracy']:.4f}
- 验证集准确率: {evaluation_results['val_results']['accuracy']:.4f}
- 合并数据集准确率: {evaluation_results['merged_results']['accuracy']:.4f}

## 表现最佳的类别 (前 3 名)
- 训练集: {', '.join([f"类别 {i}" for i in np.argsort(evaluation_results['train_results']['per_class_accuracy'])[-3:][::-1]])}
- 测试集: {', '.join([f"类别 {i}" for i in np.argsort(evaluation_results['test_results']['per_class_accuracy'])[-3:][::-1]])}
- 验证集: {', '.join([f"类别 {i}" for i in np.argsort(evaluation_results['val_results']['per_class_accuracy'])[-3:][::-1]])}

## 表现较差的类别 (后 3 名)
- 训练集: {', '.join([f"类别 {i}" for i in np.argsort(evaluation_results['train_results']['per_class_accuracy'])[:3]])}
- 测试集: {', '.join([f"类别 {i}" for i in np.argsort(evaluation_results['test_results']['per_class_accuracy'])[:3]])}
- 验证集: {', '.join([f"类别 {i}" for i in np.argsort(evaluation_results['val_results']['per_class_accuracy'])[:3]])}

## 备注
- 该模型已在训练集、测试集、验证集和合并数据集上进行评估
- 详细的评估结果、混淆矩阵和可视化结果存储在 evaluation_results 目录中
- 最佳模型已保存，可用于推理

生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

summary_file = os.path.join(SAVE_PATH, 'model_summary.md')
with open(summary_file, 'w') as f:
    f.write(summary_report)
print(f"模型训练与评估摘要已保存: {summary_file}")

# 分析最佳模型的特征重要性
print("\n正在分析特征重要性...")
feature_importance = analyze_model_features(
    best_model,
    save_path=os.path.join(training_info_dir, 'feature_importance.png'),
    apply_pca_flag=APPLY_PCA,
    pca_model=pca_model
)

print("\n训练与评估完成！")
print(f"所有结果已保存至: {SAVE_PATH}")

In [ ]:
# 可视化混淆矩阵
plt.figure(figsize=(20, 16))
conf_matrix = validation_results['confusion_matrix']

# 使用log scale可以更好地显示大量类别的混淆矩阵
plt.imshow(np.log1p(conf_matrix), cmap='Blues')
plt.colorbar(label='log(频次)')
plt.xlabel('预测类别')
plt.ylabel('真实类别')
plt.title('多分类混淆矩阵 (log scale)')

# 保存图像
plt.savefig(os.path.join(SAVE_PATH, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()

# 可视化训练过程
plt.figure(figsize=(15, 5))

# 绘制损失曲线
plt.subplot(1, 2, 1)
plt.plot(range(len(training_results['loss_list'])), training_results['loss_list'])
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)

# 绘制准确率曲线
plt.subplot(1, 2, 2)
plt.plot(range(len(training_results['acc_list'])), training_results['acc_list'], label='Train Acc')
plt.plot(training_results['val_epoch_list'], training_results['val_acc_list'], label='Val Acc')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, 'training_curves.png'))
plt.show()

In [ ]:
# 保存验证报告
validation_report = f"""
# 验证报告 - 多分类KAN (共{NUM_CLASS}类)

## 训练信息
- 训练时间: {training_results['train_time']:.2f} 秒
- 总训练轮数: {len(training_results['loss_list'])}
- 最佳模型: {os.path.basename(best_model_path)}
- 学习率: {LR}
- 批量大小: {BATCH_SIZE}
- 固定网格大小: {FIXED_GRID}

## 性能指标
- 验证集准确率: {validation_results['accuracy']:.4f}
- 最佳类别准确率: {np.max(validation_results['per_class_accuracy']):.4f} (类别 {np.argmax(validation_results['per_class_accuracy'])})
- 最差类别准确率: {np.min(validation_results['per_class_accuracy']):.4f} (类别 {np.argmin(validation_results['per_class_accuracy'])})
- 平均类别准确率: {np.mean(validation_results['per_class_accuracy']):.4f}

## 分类报告
{validation_results['report']}

## 每个类别的准确率
"""

# 添加每个类别的准确率
for i, acc in enumerate(validation_results['per_class_accuracy']):
    validation_report += f"- 类别 {i}: {acc:.4f}\n"

with open(os.path.join(SAVE_PATH, 'validation_report.txt'), 'w') as f:
    f.write(validation_report)

print(f"验证报告已保存至: {os.path.join(SAVE_PATH, 'validation_report.txt')}")

In [ ]:
def evaluate_on_all_datasets(model, train_loader, test_loader, val_loader, device, save_path):
    """评估模型在所有数据集上的性能"""
    print("在所有数据集上进行评估...")
    
    # 在训练集上评估
    print("评估训练集...")
    train_results = evaluate_multiclass_model(model, train_loader, device)
    
    # 在测试集上评估
    print("评估测试集...")
    test_results = evaluate_multiclass_model(model, test_loader, device)
    
    # 在验证集上评估
    print("评估验证集...")
    val_results = evaluate_multiclass_model(model, val_loader, device)
    
    # 绘制所有数据集的准确率对比
    plt.figure(figsize=(15, 10))
    plt.suptitle(f"不同数据集性能对比", fontsize=16)
    
    # 整体准确率对比
    plt.subplot(2, 2, 1)
    datasets = ['Training', 'Testing', 'Validation']
    accuracies = [train_results['accuracy'], test_results['accuracy'], val_results['accuracy']]
    plt.bar(datasets, accuracies, color=['blue', 'green', 'orange'])
    plt.ylabel('Accuracy')
    plt.title('Overall Accuracy Comparison')
    plt.grid(axis='y')
    for i, v in enumerate(accuracies):
        plt.text(i, v + 0.01, f"{v:.4f}", ha='center')
    
    # 绘制每个类别在各个数据集上的准确率
    plt.subplot(2, 2, 2)
    x = np.arange(NUM_CLASS)
    width = 0.25
    
    plt.bar(x - width, train_results['per_class_accuracy'], width, label='Training')
    plt.bar(x, test_results['per_class_accuracy'], width, label='Testing')
    plt.bar(x + width, val_results['per_class_accuracy'], width, label='Validation')
    
    plt.xlabel('Class')
    plt.ylabel('Accuracy')
    plt.title('Per-Class Accuracy Across Datasets')
    plt.xticks(x, [str(i) for i in range(NUM_CLASS)], rotation=90)
    plt.legend()
    
    # 保存图片
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(os.path.join(save_path, 'dataset_comparison.png'))
    plt.show()
    
    # 创建数据集比较报告
    comparison_report = f"""
    # 数据集评估性能对比报告 - 多分类KAN (共{NUM_CLASS}类)

    ## 整体准确率
    | 数据集 | 准确率 |
    |--------|--------|
    | 训练集 | {train_results['accuracy']:.4f} |
    | 测试集 | {test_results['accuracy']:.4f} |
    | 验证集 | {val_results['accuracy']:.4f} |
    
    ## 类别分布
    | 数据集 | 总样本数 | 每个类别的平均样本数 |
    |--------|----------|----------------------|
    | 训练集 | {len(train_loader.dataset)} | {len(train_loader.dataset) / NUM_CLASS:.1f} |
    | 测试集 | {len(test_loader.dataset)} | {len(test_loader.dataset) / NUM_CLASS:.1f} |
    | 验证集 | {len(val_loader.dataset)} | {len(val_loader.dataset) / NUM_CLASS:.1f} |
    
    ## 性能差异分析
    - 训练集与测试集准确率差异: {abs(train_results['accuracy'] - test_results['accuracy']):.4f}
    - 训练集与验证集准确率差异: {abs(train_results['accuracy'] - val_results['accuracy']):.4f}
    - 测试集与验证集准确率差异: {abs(test_results['accuracy'] - val_results['accuracy']):.4f}
    
    ## 类别性能分析
    - 训练集最佳类别: {np.argmax(train_results['per_class_accuracy'])} (准确率: {np.max(train_results['per_class_accuracy']):.4f})
    - 训练集最差类别: {np.argmin(train_results['per_class_accuracy'])} (准确率: {np.min(train_results['per_class_accuracy']):.4f})
    - 测试集最佳类别: {np.argmax(test_results['per_class_accuracy'])} (准确率: {np.max(test_results['per_class_accuracy']):.4f})
    - 测试集最差类别: {np.argmin(test_results['per_class_accuracy'])} (准确率: {np.min(test_results['per_class_accuracy']):.4f})
    
    ## 每个类别的准确率对比
    | 类别 | 训练集准确率 | 测试集准确率 | 验证集准确率 |
    |------|------------|------------|------------|
    """
    
    for i in range(NUM_CLASS):
        comparison_report += f"| {i} | {train_results['per_class_accuracy'][i]:.4f} | {test_results['per_class_accuracy'][i]:.4f} | {val_results['per_class_accuracy'][i]:.4f} |\n"
    
    with open(os.path.join(save_path, 'dataset_comparison_report.txt'), 'w') as f:
        f.write(comparison_report)
    
    print(f"数据集比较报告已保存至: {os.path.join(save_path, 'dataset_comparison_report.txt')}")
    
    return {
        'train_results': train_results,
        'test_results': test_results,
        'val_results': val_results
    }

In [ ]:
# def predict_multiclass(model, data, device, apply_pca=True, pca_model=None, norm=True):
#     """
#     使用训练好的模型进行多分类预测
    
#     参数:
#         model: 训练好的KAN模型
#         data: 输入数据
#         device: 计算设备
#         apply_pca: 是否应用PCA
#         pca_model: PCA模型
#         norm: 是否进行归一化
        
#     返回:
#         predictions: 预测的类别
#         probabilities: 每个类别的概率
#     """
#     model.eval()
    
#     # 应用PCA和归一化（如果需要）
#     if apply_pca and pca_model is not None:
#         data = pca_model.transform(data)
#         if norm:
#             data = (data - np.min(data, axis=0)) / (np.max(data, axis=0) - np.min(data, axis=0) + 1e-10)
    
#     # 转换为tensor
#     data_tensor = torch.FloatTensor(data).to(device)
    
#     # 批处理预测
#     batch_size = 64
#     all_probs = []
#     all_preds = []
    
#     with torch.no_grad():
#         for i in range(0, len(data), batch_size):
#             batch = data_tensor[i:i+batch_size]
#             outputs = model(batch)
#             probs = torch.softmax(outputs, dim=1)
#             _, preds = torch.max(outputs, 1)
            
#             all_probs.append(probs.cpu().numpy())
#             all_preds.append(preds.cpu().numpy())
    
#     all_probs = np.vstack(all_probs)
#     all_preds = np.concatenate(all_preds)
    
#     return all_preds, all_probs